# Text generation with pretrained GPT-2

This notebook demonstrates one thing: **how autoregressive generation works after a GPT model has already been pretrained**.

It is not a training notebook. It downloads GPT-2 weights, loads them into either minGPT or Hugging Face Transformers, then repeatedly predicts the next token from a prompt.

**Disk warning:** `from_pretrained(model_type)` downloads model files into the Hugging Face cache. Keep `model_type = 'gpt2'` unless you intentionally want a much larger model. `gpt2-xl` can consume many GB of disk space and is unnecessary for learning this notebook.

In [ ]:
import torch
from mingpt.model import GPT
from mingpt.utils import set_seed
from mingpt.bpe import BPETokenizer

def require_transformers():
    try:
        from transformers import GPT2Tokenizer, GPT2LMHeadModel
        return GPT2Tokenizer, GPT2LMHeadModel
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "generate.ipynb needs the optional Hugging Face dependency `transformers` "
            "to load pretrained GPT-2 weights. Install it in this notebook kernel with:\n\n"
            "%pip install transformers\n"
        ) from exc

set_seed(3407)

In [ ]:
use_mingpt = True # use minGPT or huggingface/transformers model?
model_type = 'gpt2' # safe default; do not use 'gpt2-xl' unless you intentionally want a large download
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Safety guard: this notebook is for learning generation mechanics, not benchmarking huge models.
allow_large_models = False
large_models = {'gpt2-medium', 'gpt2-large', 'gpt2-xl'}

In [ ]:
if model_type in large_models and not allow_large_models:
    raise ValueError(
        f"Refusing to download large model {model_type!r}. "
        "Keep model_type='gpt2' for this learning notebook, or set allow_large_models=True intentionally."
    )

GPT2Tokenizer, GPT2LMHeadModel = require_transformers()

if use_mingpt:
    model = GPT.from_pretrained(model_type)
else:
    model = GPT2LMHeadModel.from_pretrained(model_type)
    model.config.pad_token_id = model.config.eos_token_id # suppress a warning

# ship model to device and set to eval mode
model.to(device)
model.eval();

In [ ]:

def generate(prompt='', num_samples=10, steps=20, do_sample=True):
        
    # tokenize the input prompt into integer input sequence
    if use_mingpt:
        tokenizer = BPETokenizer()
        if prompt == '':
            # to create unconditional samples...
            # manually create a tensor with only the special <|endoftext|> token
            # similar to what openai's code does here https://github.com/openai/gpt-2/blob/master/src/generate_unconditional_samples.py
            x = torch.tensor([[tokenizer.encoder.encoder['<|endoftext|>']]], dtype=torch.long)
        else:
            x = tokenizer(prompt).to(device)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained(model_type)
        if prompt == '': 
            # to create unconditional samples...
            # huggingface/transformers tokenizer special cases these strings
            prompt = '<|endoftext|>'
        encoded_input = tokenizer(prompt, return_tensors='pt').to(device)
        x = encoded_input['input_ids']
    
    # we'll process all desired num_samples in a batch, so expand out the batch dim
    x = x.expand(num_samples, -1)

    # forward the model `steps` times to get samples, in a batch
    y = model.generate(x, max_new_tokens=steps, do_sample=do_sample, top_k=40)
    
    for i in range(num_samples):
        out = tokenizer.decode(y[i].cpu().squeeze())
        print('-'*80)
        print(out)
        

In [ ]:
generate(prompt='Andrej Karpathy, the', num_samples=3, steps=20)